# ManimFlow

Autonomous animation generator powered by [Aider](https://aider.chat) — **no API keys required**.

## Usage
1. Runtime → Run All (first run installs deps and asks you to restart)
2. Runtime → Restart session, then Runtime → Run All again
3. Describe your animation in the text box that appears in the last cell
4. Click **Generate**

In [ ]:
import importlib.util, pathlib, sys, subprocess, platform, os

if platform.system() == 'Linux':
    if subprocess.run(['dpkg', '-s', 'libpango1.0-dev'], capture_output=True).returncode != 0:
        print("Installing system deps...")
        subprocess.run(
            ['apt-get', 'install', '-y', '-q',
             'libcairo2-dev', 'libpango1.0-dev', 'ffmpeg',
             'texlive-latex-base', 'texlive-latex-extra',
             'texlive-fonts-recommended', 'texlive-plain-generic', 'dvisvgm'],
            check=True
        )
elif platform.system() == 'Darwin':
    sdk = subprocess.run(['xcrun', '--show-sdk-path'], capture_output=True, text=True).stdout.strip()
    if sdk:
        os.environ.setdefault('SDKROOT', sdk)
    for pkg in ['cairo', 'pango', 'ffmpeg']:
        if subprocess.run(['brew', 'list', pkg], capture_output=True).returncode != 0:
            subprocess.run(['brew', 'install', pkg])

# Colab ships numpy 2.x which breaks scipy/manim — pin to compatible versions
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.26.4', 'scipy==1.13.1'],
    check=True
)

if not importlib.util.find_spec('manim'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'manim', '-q'], check=True)

if not importlib.util.find_spec('aider'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'aider-chat', '-q'], check=True)

print("Deps installed. Please restart the runtime (Runtime → Restart session), then run all cells again.")

In [ ]:
import pathlib, subprocess, re, json, threading, urllib.request
from http.server import HTTPServer, BaseHTTPRequestHandler
from socketserver import ThreadingMixIn

BASE = pathlib.Path.cwd()
MANIM_OUTPUT = BASE / 'manim_output'
MANIM_OUTPUT.mkdir(parents=True, exist_ok=True)

EXA_URL = "https://demos.exa.ai/chatbot-demo/api/chat/stream"
AGENT_SYSTEM_PROMPT = """\
You are a precise assistant. Follow the user's instructions exactly.
- Never add language tags to code fences — use plain triple backticks only: ``` not ```python or ```bash
- Never use heredocs (cat << EOF). Write files with python3 -c instead.
- Write your entire shell command on a single line — no literal newlines inside the code fence.
- No follow-up suggestions, no lists of questions. Be concise and direct."""

def _fix_encoding(text):
    return re.sub(r'Â([\x80-\xBF])', lambda m: m.group(1), text)

def _strip_followups(text):
    idx = text.find("\n\n```followups")
    if idx >= 0:
        text = text[:idx]
    text = re.sub(r'\n\[(["\']).*?\1(?:,\s*(["\']).*?\2)*\s*\]\s*$', '', text, flags=re.DOTALL)
    return text.rstrip()

def _strip_fence_tags(text):
    text = re.sub(r'^`{3,}[a-zA-Z_][a-zA-Z0-9_]*$', '```', text, flags=re.MULTILINE)
    text = re.sub(r'^`{4,}$', '```', text, flags=re.MULTILINE)
    return text

def _clean_code_fences(text):
    last = text.rfind("```\n")
    if last == -1:
        return text
    before = text[:last]
    after = text[last + 4:]
    close = after.rfind("\n```")
    if close == -1:
        return text
    body = after[:close]
    rest = after[close + 4:]
    body = re.sub(r'^`+\n?', '', body)
    if re.search(r'python3\s+-c', body):
        body = body.replace('\n', '\\n')
    else:
        body = body.split('\n')[0]
    return before + "```\n" + body + "\n```" + rest

def _rewrite_heredocs(text):
    def replace(m):
        filepath = m.group(2)
        content = m.group(3)
        escaped = content.replace('\\', '\\\\').replace("'", "\\'").replace('\n', '\\n')
        return (f"python3 -c \"import pathlib; "
                f"pathlib.Path('{filepath}').parent.mkdir(parents=True, exist_ok=True); "
                f"pathlib.Path('{filepath}').write_text('{escaped}')\"")
    return re.sub(r"cat\s*<<\s*['\"]?(\w+)['\"]?\s*>\s*(\S+)\n([\s\S]*?)\n\1", replace, text)

def _post_process(text):
    text = _fix_encoding(text)
    text = _strip_followups(text)
    text = _strip_fence_tags(text)
    text = _rewrite_heredocs(text)
    text = _clean_code_fences(text)
    return text

def _collect_exa_stream(messages):
    system_msg = next((m for m in messages if m['role'] == 'system'), None)
    non_system = [m for m in messages if m['role'] != 'system']
    last = non_system[-1]
    history = [{'role': m['role'], 'content': m['content']} for m in non_system[:-1]]
    system_content = system_msg['content'] if system_msg else AGENT_SYSTEM_PROMPT
    payload = json.dumps({
        'message': f"{system_content}\n\n{last['content']}",
        'history': history,
        'exaEnabled': False,
        'model': 'google/gemini-2.5-flash',
        'searchType': 'instant',
    }).encode()
    req = urllib.request.Request(EXA_URL, data=payload,
        headers={'Content-Type': 'application/json', 'Accept': 'text/event-stream'})
    full = ''
    current_event = None
    with urllib.request.urlopen(req, timeout=120) as resp:
        buf = b''
        for raw in resp:
            buf += raw
            lines = buf.split(b'\n')
            buf = lines.pop()
            for line in lines:
                t = line.decode('utf-8', errors='replace').strip()
                if not t:
                    continue
                if t.startswith('event:'):
                    current_event = t[6:].strip()
                elif t.startswith('data:') and current_event == 'content':
                    try:
                        full += json.loads(t[5:].strip()).get('content', '')
                    except Exception:
                        pass
    return _post_process(full)

LOCAL_PORT = 18642

class _ThreadingServer(ThreadingMixIn, HTTPServer):
    daemon_threads = True

class _Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def do_GET(self):
        if self.path == '/v1/models':
            body = json.dumps({'object': 'list', 'data': [
                {'id': 'manimator', 'object': 'model', 'created': 0, 'owned_by': 'exa'}
            ]}).encode()
            self.send_response(200)
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(body)
        else:
            self.send_response(404)
            self.end_headers()

    def do_POST(self):
        if self.path != '/v1/chat/completions':
            self.send_response(404)
            self.end_headers()
            return
        length = int(self.headers.get('Content-Length', 0))
        body = json.loads(self.rfile.read(length))
        messages = body.get('messages', [])
        try:
            content = _collect_exa_stream(messages)
        except Exception as e:
            self.send_response(500)
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'error': str(e)}).encode())
            return
        resp_body = json.dumps({
            'id': 'chatcmpl-local',
            'object': 'chat.completion',
            'model': 'manimator',
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant', 'content': content},
                'finish_reason': 'stop',
            }],
            'usage': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0},
        }).encode()
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(resp_body)))
        self.end_headers()
        self.wfile.write(resp_body)

def _start_server():
    try:
        server = _ThreadingServer(('127.0.0.1', LOCAL_PORT), _Handler)
    except OSError:
        print(f'Port {LOCAL_PORT} already in use — server already running')
        return None
    t = threading.Thread(target=server.serve_forever, daemon=True)
    t.start()
    return server

_start_server()
print(f'Local LLM server running on port {LOCAL_PORT}')
print('Setup complete')

In [ ]:
import subprocess, pathlib, shutil, os, json, urllib.request
import IPython.display

def call_llm(messages):
    payload = json.dumps({
        'model': 'manimator',
        'messages': messages,
        'max_tokens': 1024,
    }).encode()
    req = urllib.request.Request(
        f'http://127.0.0.1:{LOCAL_PORT}/v1/chat/completions',
        data=payload,
        headers={'Content-Type': 'application/json', 'Authorization': 'Bearer dummy'},
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read())['choices'][0]['message']['content'].strip()

def plan_animation(user_prompt):
    return call_llm([
        {
            'role': 'system',
            'content': (
                'You are a Manim animation planner. Given a topic, write a concise scene plan '
                'for a short animation — 3 to 5 scenes max. For each scene describe: what shapes/colors appear, '
                'how they move, what text labels show. Be brief and specific. No prose, no markdown headers, '
                'just a numbered list. Total response under 200 words.'
            ),
        },
        {'role': 'user', 'content': user_prompt},
    ])

def run_manimator(user_prompt):
    print(f'Planning: {user_prompt!r}')
    plan = plan_animation(user_prompt)
    print('\n=== Scene Plan ===')
    print(plan)
    print('=' * 40)

    scene_file = MANIM_OUTPUT / 'scene.py'
    scene_file.unlink(missing_ok=True)

    task = (
        f"Write a Manim animation scene in scene.py implementing this plan:\n\n{plan}\n\n"
        f"The class must be named AnimScene and subclass Scene. "
        f"Use plain Text() for labels (avoid LaTeX in Text), MathTex only for equations. "
        f"Make it visually complete."
    )

    aider_env = {
        **os.environ,
        'OPENAI_API_KEY': 'dummy',
        'GIT_AUTHOR_NAME': 'aider',
        'GIT_AUTHOR_EMAIL': 'aider@colab',
        'GIT_COMMITTER_NAME': 'aider',
        'GIT_COMMITTER_EMAIL': 'aider@colab',
    }

    def run_aider(message):
        r = subprocess.run(
            [
                'aider',
                '--model', 'openai/manimator',
                '--openai-api-base', f'http://127.0.0.1:{LOCAL_PORT}/v1',
                '--openai-api-key', 'dummy',
                '--yes-always',
                '--no-stream',
                '--no-auto-commits',
                '--message', message,
                'scene.py',
            ],
            cwd=str(MANIM_OUTPUT),
            capture_output=True, text=True, timeout=300, env=aider_env,
        )
        out = r.stdout
        if r.stderr:
            out += '\n[stderr] ' + r.stderr[-300:]
        print(out[-3000:] if len(out) > 3000 else out)

    print('\n=== Aider coding... ===')
    run_aider(task)

    if not scene_file.exists():
        print('ERROR: aider did not produce scene.py')
        return

    for attempt in range(3):
        print(f'\n=== Rendering (attempt {attempt + 1}) ===')
        render = subprocess.run(
            ['python3', '-m', 'manim', '-pql', '--disable_caching',
             str(scene_file), 'AnimScene'],
            cwd=str(MANIM_OUTPUT),
            capture_output=True, text=True, timeout=300,
        )
        render_out = render.stdout + render.stderr
        print(render_out[-2000:])

        videos = sorted(MANIM_OUTPUT.rglob('*.mp4'), key=lambda p: p.stat().st_mtime)
        if videos:
            break

        if attempt < 2:
            print('\n=== Render failed — asking aider to fix ===')
            fix_task = (
                f"The Manim render of scene.py failed:\n\n{render_out[-1500:]}\n\n"
                f"Fix scene.py so it renders without errors. "
                f"If the error mentions LaTeX or tex, replace all MathTex() with Text() using plain strings. "
                f"Keep the class named AnimScene."
            )
            run_aider(fix_task)

    videos = sorted(MANIM_OUTPUT.rglob('*.mp4'), key=lambda p: p.stat().st_mtime)
    if not videos:
        print('\nNo video rendered after all attempts.')
        return

    dest = BASE / 'animation.mp4'
    shutil.copy(videos[-1], dest)

    for mp4 in videos:
        try: mp4.unlink()
        except: pass
    for d in MANIM_OUTPUT.rglob('media'):
        if d.is_dir():
            shutil.rmtree(d, ignore_errors=True)

    print(f'\nFinal MP4: {dest}')
    IPython.display.display(
        IPython.display.Video(str(dest), embed=True, html_attributes='controls width="720"')
    )

print('run_manimator() ready')

In [ ]:
import ipywidgets as widgets
from IPython.display import display

prompt_box = widgets.Textarea(
    placeholder="e.g. Explain how a Fourier series builds up a square wave",
    layout=widgets.Layout(width="700px", height="100px"),
)
btn = widgets.Button(description="Generate", button_style="primary")
status = widgets.Output()

def on_generate(b):
    with status:
        status.clear_output()
        if not prompt_box.value.strip():
            print("Please enter a prompt.")
            return
        run_manimator(prompt_box.value.strip())

btn.on_click(on_generate)
display(prompt_box, btn, status)